In [ ]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.5 MB/s eta 0:00:00


In [ ]:
import os
import random
import math
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, precision_score,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchaudio
from transformers import (
    XLMRobertaTokenizer, XLMRobertaModel,
    WhisperProcessor, WhisperModel
)

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors

# =====================================
# Reproducibility & Config
# =====================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths
CSV_PATH = "/content/new synthetic_diaz - Sheet1.csv"  # <--- change to your csv
RESULTS_DIR = "results"
CKPT_DIR = os.path.join(RESULTS_DIR, "checkpoints")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# Hyperparams
N_FOLDS = 5
BATCH_SIZE = 8
EPOCHS = 15
EARLY_PATIENCE = 4           # early stopping patience on val Fusion F1
ALPHA = 0.5                  # multi-task weighting: (text+audio) vs fusion
MAX_TEXT_LEN = 128
WHISPER_SR = 16000

# =====================================
# Dataset
# =====================================
class SuicideDataset(Dataset):
    def __init__(self, df, tokenizer, whisper_processor, max_text_len=128, sample_rate=16000):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.wp = whisper_processor
        self.max_text_len = max_text_len
        self.sample_rate = sample_rate

    def __len__(self):
        return len(self.df)

    def _load_audio(self, path):
        try:
            if not isinstance(path, str) or not os.path.exists(path):
                raise FileNotFoundError
            wav, sr = torchaudio.load(path)
            if wav.shape[0] > 1:
                wav = wav.mean(dim=0, keepdim=True)
            if sr != self.sample_rate:
                wav = torchaudio.functional.resample(wav, sr, self.sample_rate)
            return wav.squeeze(0), self.sample_rate
        except Exception:
            # 1s of silence if missing/corrupt
            return torch.zeros(self.sample_rate), self.sample_rate

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["text"]) if "text" in row else ""
        text_label = int(row["text_label"]) if "text_label" in row else 0
        audio_path = str(row["audio"]) if "audio" in row else ""
        audio_label = int(row["audio_label"]) if "audio_label" in row else text_label
        text_lang = str(row.get("text_lang", "unknown"))
        audio_lang = str(row.get("audio_lang", "unknown"))

        # Text encode
        enc = self.tok(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_text_len,
            return_tensors="pt"
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)

        # Audio features for Whisper
        wav, sr = self._load_audio(audio_path)
        proc = self.wp(wav.numpy(), sampling_rate=sr, return_tensors="pt")
        audio_features = proc["input_features"].squeeze(0)  # [80, T]

        return (
            input_ids,
            attention_mask,
            audio_features,
            torch.tensor(text_label, dtype=torch.long),
            torch.tensor(audio_label, dtype=torch.long),
            text_lang,
            audio_lang,
        )

import copy

# ============================
# Load Pretrained Encoders Once
# ============================
print("Loading pretrained encoders...")
text_encoder_pretrained = XLMRobertaModel.from_pretrained("xlm-roberta-base")
whisper_full = WhisperModel.from_pretrained("openai/whisper-small")
audio_encoder_pretrained = whisper_full.encoder
audio_dim = whisper_full.config.d_model
text_dim = text_encoder_pretrained.config.hidden_size  # 768
hidden_dim = 512  # Projection block hidden size
num_labels = 2

# Freeze Whisper
for param in audio_encoder_pretrained.parameters():
    param.requires_grad = False
print("Whisper encoder frozen.")

# ============================
# Updated Model
# ============================
class MultiTaskFusionModel(nn.Module):
    def __init__(self, text_encoder, audio_encoder, audio_dim, text_dim, hidden_dim=512, num_classes=2):
        super().__init__()
        self.text_encoder = text_encoder
        self.audio_encoder = audio_encoder

        # Audio projection to text space (for fusion)
        self.audio_proj_fusion = nn.Linear(audio_dim, text_dim)

        # Audio classifier block
        self.audio_head = nn.Sequential(
            nn.Linear(audio_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_classes)
        )

        # Text classifier
        self.text_head = nn.Linear(text_dim, num_classes)

        # Gated fusion
        self.gate = nn.Linear(2 * text_dim, text_dim)
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(0.1)
        self.norm = nn.LayerNorm(text_dim)
        self.fusion_head = nn.Linear(text_dim, num_classes)

    def forward(self, input_ids, attention_mask, audio_features):
        # Text
        t_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        t_cls = t_out.last_hidden_state[:, 0, :]  # [B, D]

        # Audio (Whisper expects [B, 80, T])
        a_out = self.audio_encoder(audio_features).last_hidden_state  # [B, T', d_audio]
        a_cls = a_out[:, 0, :]  # [B, d_audio]

        # Audio classifier
        audio_logits = self.audio_head(a_cls)

        # Projection for fusion
        a_proj = self.audio_proj_fusion(a_cls)

        # Text classifier
        text_logits = self.text_head(t_cls)

        # Fusion via gate per-dimension
        combo = torch.cat([t_cls, a_proj], dim=-1)  # [B, 2D]
        g = self.sigmoid(self.gate(combo))
        fused = g * t_cls + (1.0 - g) * a_proj
        fused = self.norm(self.dropout(fused) + fused)
        fusion_logits = self.fusion_head(fused)

        return text_logits, audio_logits, fusion_logits, g



# =====================================
# Loss & Metrics
# =====================================
ce = nn.CrossEntropyLoss()

def compute_loss(text_logits, audio_logits, fusion_logits, text_labels, audio_labels, alpha=ALPHA):
    loss_text = ce(text_logits, text_labels)
    loss_audio = ce(audio_logits, audio_labels)
    # Fusion supervised by text labels (primary task)
    loss_fusion = ce(fusion_logits, text_labels)
    return alpha * (loss_text + loss_audio) + (1 - alpha) * loss_fusion

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred, average="binary", zero_division=0)
    f1  = f1_score(y_true, y_pred, average="binary", zero_division=0)
    prec= precision_score(y_true, y_pred, average="binary", zero_division=0)
    return acc, rec, f1, prec

# =====================================
# Logging helpers
# =====================================
def save_cm(y_true, y_pred, title, path, labels=(0,1), cmap="Blues"):
    acc, rec, f1, prec = compute_metrics(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=list(labels))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(cmap=cmap)
    plt.title(f"{title}\nAcc={acc:.3f} | Rec={rec:.3f} | F1={f1:.3f} | Prec={prec:.3f}")
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    return cm

def averaged_cm_image(cm, title, path, labels=(0,1), cmap="Blues"):
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(cmap=cmap)
    plt.title(title)
    plt.savefig(path, bbox_inches="tight")
    plt.close()

# =====================================
# Early Stopping
# =====================================
class EarlyStopper:
    def __init__(self, patience=2, mode="max"):
        self.patience = patience
        self.mode = mode
        self.best = None
        self.bad_epochs = 0
        self.better = (lambda cur, best: cur > best) if mode == "max" else (lambda cur, best: cur < best)

    def step(self, metric):
        if self.best is None or self.better(metric, self.best):
            self.best = metric
            self.bad_epochs = 0
            return True
        self.bad_epochs += 1
        return False

    def should_stop(self):
        return self.bad_epochs >= self.patience

# =====================================
# Epoch Loops
# =====================================
def train_one_epoch(model, loader, optimizer, epoch_idx, fold_idx, device):
    model.train()
    # Collect predictions (for quick epoch metrics on fusion)
    f_preds, f_labels = [], []

    for b_idx, batch in enumerate(loader, 1):
        ids, mask, aud, t_lab, a_lab, t_lang, a_lang = batch
        ids, mask, aud = ids.to(device), mask.to(device), aud.to(device)
        t_lab, a_lab = t_lab.to(device), a_lab.to(device)

        optimizer.zero_grad(set_to_none=True)
        t_log, a_log, f_log, g = model(ids, mask, aud)
        loss = compute_loss(t_log, a_log, f_log, t_lab, a_lab)
        loss.backward()
        optimizer.step()

        # track fusion epoch metrics (class preds)
        f_preds.extend(torch.argmax(f_log, dim=1).detach().cpu().numpy())
        f_labels.extend(t_lab.detach().cpu().numpy())

    acc, rec, f1, prec = compute_metrics(f_labels, f_preds)
    return {"acc": acc, "rec": rec, "f1": f1, "prec": prec}

def validate(model, loader, device):
    model.eval()
    all_t_preds, all_t_labels = [], []
    all_a_preds, all_a_labels = [], []
    all_f_preds, all_f_labels = [], []
    fusion_probs = []  # for AUCs (positive class prob)
    text_langs = []
    all_gates = []

    softmax = nn.Softmax(dim=1)

    with torch.no_grad():
        for batch in loader:
            ids, mask, aud, t_lab, a_lab, t_lang, a_lang = batch
            ids, mask, aud = ids.to(device), mask.to(device), aud.to(device)

            t_log, a_log, f_log, g = model(ids, mask, aud)

            # predictions
            t_pred = torch.argmax(t_log, dim=1).cpu().numpy()
            a_pred = torch.argmax(a_log, dim=1).cpu().numpy()
            f_pred = torch.argmax(f_log, dim=1).cpu().numpy()

            # probs for AUC (fusion)
            f_prob = softmax(f_log)[:, 1].cpu().numpy()

            all_t_preds.extend(t_pred)
            all_a_preds.extend(a_pred)
            all_f_preds.extend(f_pred)

            all_t_labels.extend(t_lab.numpy())
            all_a_labels.extend(a_lab.numpy())
            all_f_labels.extend(t_lab.numpy())  # fusion evaluated vs text labels

            fusion_probs.extend(f_prob)
            text_langs.extend(list(t_lang))
            all_gates.append(g.detach().cpu().numpy())

    # Fusion metrics
    acc, rec, f1, prec = compute_metrics(all_f_labels, all_f_preds)

    all_gates = np.concatenate(all_gates, axis=0).flatten() if len(all_gates) else np.array([])
    return {
        "acc": acc, "rec": rec, "f1": f1, "prec": prec,
        "t_preds": np.array(all_t_preds), "t_labels": np.array(all_t_labels),
        "a_preds": np.array(all_a_preds), "a_labels": np.array(all_a_labels),
        "f_preds": np.array(all_f_preds), "f_labels": np.array(all_f_labels),
        "f_probs": np.array(fusion_probs),
        "text_langs": np.array(text_langs),
        "gates": all_gates
    }

# =====================================
# Per-language CMs & Text vs Audio agreement
# =====================================
def per_language_cm_text(val_res, fold_idx, out_dir=RESULTS_DIR):
    imgs = []
    langs = np.array([str(x).lower() for x in val_res["text_langs"]])
    is_en = np.array([("en" in x) or ("english" in x) for x in langs])
    is_hi = np.array([("hi" in x) or ("hindi" in x) for x in langs])

    cm_sum = {"english": np.zeros((2,2), dtype=int), "hindi": np.zeros((2,2), dtype=int)}

    def do_subset(mask, name, cmap):
        if mask.sum() == 0:
            return None, np.zeros((2,2), dtype=int)
        y_true = val_res["t_labels"][mask]
        y_pred = val_res["t_preds"][mask]
        p = os.path.join(out_dir, f"fold{fold_idx}_text_{name}_cm.png")
        cm = save_cm(y_true, y_pred, f"Fold {fold_idx} - Text Head ({name.title()})", p, labels=(0,1), cmap=cmap)
        return p, cm

    p_en, cm_en = do_subset(is_en, "english", "Greens")
    p_hi, cm_hi = do_subset(is_hi, "hindi", "Purples")
    if p_en: imgs.append(p_en)
    if p_hi: imgs.append(p_hi)

    cm_sum["english"] += cm_en
    cm_sum["hindi"] += cm_hi
    return imgs, cm_sum

def text_vs_audio_cm(val_res, fold_idx, out_dir=RESULTS_DIR):
    agree_pred = (val_res["t_preds"] == val_res["a_preds"]).astype(int)
    agree_true = (val_res["t_labels"] == val_res["a_labels"]).astype(int)
    p = os.path.join(out_dir, f"fold{fold_idx}_text_vs_audio_cm.png")
    cm = save_cm(agree_true, agree_pred, f"Fold {fold_idx} - Text vs Audio Agreement", p, labels=(0,1), cmap="Oranges")
    return p, cm

# =====================================
# PDF Export (existing minimal per-epoch summary)
# =====================================
def export_pdf(per_epoch_rows, per_fold_images, final_images, out_path):
    styles = getSampleStyleSheet()
    doc = SimpleDocTemplate(out_path, pagesize=A4)
    elems = [Paragraph("Multimodal Suicide Prediction Report", styles["Title"]), Spacer(1, 12)]

    # Per-epoch summary lines (fusion val)
    for (fold, epoch, acc, rec, f1, prec) in per_epoch_rows:
        elems.append(Paragraph(f"Fold {fold} — Epoch {epoch}: Acc={acc:.4f} | Recall={rec:.4f} | F1={f1:.4f} | Prec={prec:.4f}", styles["Normal"]))
    elems.append(Spacer(1, 16))

    # Per-fold images
    for img in per_fold_images:
        elems.append(RLImage(img, width=420, height=320))
        elems.append(Spacer(1, 10))

    # Final summary images
    elems.append(Spacer(1, 16))
    elems.append(Paragraph("Final Summary (Averaged Across Folds)", styles["Heading2"]))
    for img in final_images:
        elems.append(RLImage(img, width=420, height=320))
        elems.append(Spacer(1, 10))

    doc.build(elems)
    print(f"PDF saved at {out_path}")

# =====================================
# Final Report (rich, per your new asks)
# =====================================
def generate_final_report(
    out_file="results/final_report.pdf",
    n_folds=5,
    gate_means=None,
    gate_vars=None,
    fusion_precisions=None,
    fusion_roc_aucs=None,
    fusion_pr_aucs=None,
    per_fold_metric_table=None,   # list of dicts for each fold
):
    """
    Generate a comprehensive PDF report of all folds.
    Includes metrics, confusion matrices, gate histograms, ROC/PR curves, and aggregated stats.
    """
    styles = getSampleStyleSheet()
    styleH = styles["Heading1"]
    styleH2 = styles["Heading2"]
    styleN = styles["Normal"]

    Story = []
    Story.append(Paragraph("Multimodal Suicide Detection - Cross Validation Report", styleH))
    Story.append(Spacer(1, 12))

    # Per Fold Results
    for fold in range(1, n_folds + 1):
        Story.append(Paragraph(f"Fold {fold} Results", styleH2))
        Story.append(Spacer(1, 8))

        # CMs
        for head in ["text", "audio", "fusion"]:
            cm_file = f"results/fold{fold}_{head}_cm.png"
            if os.path.exists(cm_file):
                Story.append(Paragraph(f"{head.title()} Confusion Matrix", styleN))
                Story.append(RLImage(cm_file, width=300, height=300))
                Story.append(Spacer(1, 8))

        # Per-language text CMs (if present)
        for name in ["english", "hindi"]:
            cm_file = f"results/fold{fold}_text_{name}_cm.png"
            if os.path.exists(cm_file):
                Story.append(Paragraph(f"Text ({name.title()}) Confusion Matrix", styleN))
                Story.append(RLImage(cm_file, width=300, height=300))
                Story.append(Spacer(1, 8))

        # Text vs Audio agreement
        tvsa = f"results/fold{fold}_text_vs_audio_cm.png"
        if os.path.exists(tvsa):
            Story.append(Paragraph("Text vs Audio Agreement", styleN))
            Story.append(RLImage(tvsa, width=300, height=300))
            Story.append(Spacer(1, 8))

        # Gate histogram
        gh_file = f"results/fusion_gate_hist_fold{fold}.png"
        if os.path.exists(gh_file):
            Story.append(Paragraph("Fusion Gate Histogram", styleN))
            Story.append(RLImage(gh_file, width=300, height=220))
            Story.append(Spacer(1, 8))

        # ROC / PR curves
        roc_file = f"results/roc_fold{fold}.png"
        pr_file  = f"results/pr_fold{fold}.png"
        if os.path.exists(roc_file):
            Story.append(Paragraph("ROC Curve (Fusion)", styleN))
            Story.append(RLImage(roc_file, width=320, height=240))
            Story.append(Spacer(1, 8))
        if os.path.exists(pr_file):
            Story.append(Paragraph("Precision-Recall Curve (Fusion)", styleN))
            Story.append(RLImage(pr_file, width=320, height=240))
            Story.append(Spacer(1, 8))

        # Metrics Table for fold
        if per_fold_metric_table is not None:
            m = per_fold_metric_table[fold-1]
            data = [
                ["Metric", "Text", "Audio", "Fusion"],
                ["Accuracy", f"{m['text_acc']:.4f}", f"{m['audio_acc']:.4f}", f"{m['fusion_acc']:.4f}"],
                ["Recall",   f"{m['text_rec']:.4f}", f"{m['audio_rec']:.4f}", f"{m['fusion_rec']:.4f}"],
                ["F1",       f"{m['text_f1']:.4f}",  f"{m['audio_f1']:.4f}",  f"{m['fusion_f1']:.4f}"],
                ["Precision",f"{m['text_prec']:.4f}",f"{m['audio_prec']:.4f}",f"{m['fusion_prec']:.4f}"],
                ["ROC AUC",  "-", "-", f"{m['fusion_roc_auc']:.4f}"],
                ["PR AUC",   "-", "-", f"{m['fusion_pr_auc']:.4f}"],
                ["Gate Mean","-", "-", f"{m['gate_mean']:.4f}"],
                ["Gate Var", "-", "-", f"{m['gate_var']:.4f}"],
            ]
            t = Table(data, hAlign="LEFT")
            t.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
                ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
                ('GRID', (0, 0), (-1, -1), 0.5, colors.black),
            ]))
            Story.append(t)
            Story.append(Spacer(1, 20))

    # Aggregated Results Section
    Story.append(Paragraph("Aggregated Results Across Folds", styleH))
    Story.append(Spacer(1, 12))

    if (gate_means is not None) and (fusion_precisions is not None) and (fusion_roc_aucs is not None) and (fusion_pr_aucs is not None):
        agg_data = [
            ["Metric", "Mean ± Std"],
            ["Gate Mean", f"{np.mean(gate_means):.4f} ± {np.std(gate_means):.4f}"],
            ["Gate Variance", f"{np.mean(gate_vars):.4f} ± {np.std(gate_vars):.4f}"],
            ["Fusion Precision", f"{np.mean(fusion_precisions):.4f} ± {np.std(fusion_precisions):.4f}"],
            ["Fusion ROC AUC", f"{np.mean(fusion_roc_aucs):.4f} ± {np.std(fusion_roc_aucs):.4f}"],
            ["Fusion PR AUC", f"{np.mean(fusion_pr_aucs):.4f} ± {np.std(fusion_pr_aucs):.4f}"],
        ]
        agg_table = Table(agg_data, hAlign="LEFT")
        agg_table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
            ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
            ('GRID', (0, 0), (-1, -1), 0.5, colors.black),
        ]))
        Story.append(agg_table)
        Story.append(Spacer(1, 20))

    # Overall CMs (if saved)
    for name, fname in [
        ("Overall Text CM", "results/overall_text_cm.png"),
        ("Overall Audio CM", "results/overall_audio_cm.png"),
        ("Overall Fusion CM", "results/overall_fusion_cm.png"),
        ("Averaged Text vs Audio Agreement (5 folds)", "results/avg_text_vs_audio_cm.png"),
        ("Averaged Text Head CM — English (5 folds)", "results/avg_text_english_cm.png"),
        ("Averaged Text Head CM — Hindi (5 folds)", "results/avg_text_hindi_cm.png"),
        ("Averaged Global Metrics Table", "results/avg_global_metrics.png"),
        ("Mean ROC Curve (Across Folds)", "results/roc_mean_across_folds.png"),
        ("Mean PR Curve (Across Folds)", "results/pr_mean_across_folds.png"),
    ]:
        if os.path.exists(fname):
            Story.append(Paragraph(name, styleH2))
            Story.append(RLImage(fname, width=420, height=320))
            Story.append(Spacer(1, 20))

    # Build PDF
    doc = SimpleDocTemplate(out_file, pagesize=(800, 1000))
    doc.build(Story)
    print(f"✅ Final PDF report saved at {out_file}")

# =====================================
# Main 5-Fold Training
# =====================================
if __name__ == "__main__":
    # Load data
    df = pd.read_csv(CSV_PATH)
    needed = ["text", "text_lang", "text_label", "audio", "audio_lang", "audio_label"]
    miss = [c for c in needed if c not in df.columns]
    if miss:
        raise ValueError(f"Missing required columns in CSV: {miss}")

    tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")
    whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    per_epoch_rows = []       # (fold, epoch, acc, rec, f1, prec) for fusion/val
    per_fold_images = []      # list of image paths (per-fold CMs etc.)
    final_images = []         # aggregated images

    # For final averaged metrics
    metrics_acc = {"text": [], "audio": [], "fusion": []}
    metrics_rec = {"text": [], "audio": [], "fusion": []}
    metrics_f1  = {"text": [], "audio": [], "fusion": []}
    metrics_prec= {"text": [], "audio": [], "fusion": []}

    # Fusion AUCs across folds
    fusion_roc_aucs = []
    fusion_pr_aucs  = []

    # Gate stats across folds
    gate_means = []
    gate_vars  = []

    # For averaged confusion matrices
    sum_cm_text_vs_audio = np.zeros((2,2), dtype=int)
    sum_cm_text_en = np.zeros((2,2), dtype=int)
    sum_cm_text_hi = np.zeros((2,2), dtype=int)
    sum_cm_text  = np.zeros((2,2), dtype=int)
    sum_cm_audio = np.zeros((2,2), dtype=int)
    sum_cm_fusion= np.zeros((2,2), dtype=int)

    # For mean ROC/PR curves (interpolate)
    mean_fpr = np.linspace(0, 1, 200)
    roc_tprs = []
    pr_recalls = np.linspace(0, 1, 200)
    pr_precisions_interp = []

    # Per-fold table to feed final report
    per_fold_metric_table = []

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df["text"].values, df["text_label"].values), 1):
        print(f"Fold {fold_idx} / {N_FOLDS} =====")

        train_df, val_df = df.iloc[train_idx], df.iloc[val_idx]

        train_ds = SuicideDataset(train_df, tokenizer, whisper_processor, max_text_len=MAX_TEXT_LEN, sample_rate=WHISPER_SR)
        val_ds   = SuicideDataset(val_df,   tokenizer, whisper_processor, max_text_len=MAX_TEXT_LEN, sample_rate=WHISPER_SR)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)



        model = MultiTaskFusionModel(
        text_encoder=copy.deepcopy(text_encoder_pretrained),
        audio_encoder=copy.deepcopy(audio_encoder_pretrained),
        audio_dim=audio_dim,
        text_dim=text_dim,
        hidden_dim=hidden_dim,
        num_classes=num_labels
    ).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
        early = EarlyStopper(patience=EARLY_PATIENCE, mode="max")

        best_ckpt_path = os.path.join(CKPT_DIR, f"fold{fold_idx}_best.pt")

        # Train epochs with early stopping
        for epoch in range(1, EPOCHS + 1):
            train_res = train_one_epoch(model, train_loader, optimizer, epoch, fold_idx, device)
            val_res = validate(model, val_loader, device)

            # Record epoch metrics (fusion on val)
            per_epoch_rows.append((fold_idx, epoch, val_res["acc"], val_res["rec"], val_res["f1"], val_res["prec"]))
            print(f"Epoch {epoch:02d} | Val Acc={val_res['acc']:.4f} | Val Rec={val_res['rec']:.4f} | Val F1={val_res['f1']:.4f} | Val Prec={val_res['prec']:.4f}")

            # Early stopping on fusion F1
            if early.step(val_res["f1"]):
                torch.save(model.state_dict(), best_ckpt_path)
            if early.should_stop():
                print("⏹️ Early stopping triggered.")
                break

        # Load best ckpt and do final fold evaluation
        if os.path.exists(best_ckpt_path):
            model.load_state_dict(torch.load(best_ckpt_path, map_location=device))
            print(f"Loaded best checkpoint: {best_ckpt_path}")
        final_val = validate(model, val_loader, device)

        # ------ Gate stats & single histogram per fold ------
        gates = final_val["gates"]
        if gates.size > 0:
            g_mean = float(gates.mean())
            g_var  = float(gates.var())
        else:
            g_mean, g_var = 0.0, 0.0
        gate_means.append(g_mean)
        gate_vars.append(g_var)

        plt.hist(gates, bins=20, range=(0,1))
        plt.title(f"Gate Histogram - Fold {fold_idx}\nMean={g_mean:.4f} | Var={g_var:.4f}")
        plt.xlabel("Gate (0=Audio, 1=Text)")
        plt.ylabel("Count")
        gh_path = os.path.join(RESULTS_DIR, f"fusion_gate_hist_fold{fold_idx}.png")
        plt.savefig(gh_path, bbox_inches="tight")
        plt.close()
        per_fold_images.append(gh_path)

        # ------ Per-fold confusion matrices (Text/Audio/Fusion) ------
        for head, labels, preds, cmap in [
            ("text",  final_val["t_labels"], final_val["t_preds"], "Greens"),
            ("audio", final_val["a_labels"], final_val["a_preds"], "Purples"),
            ("fusion",final_val["f_labels"], final_val["f_preds"], "Blues"),
        ]:
            cm_path = os.path.join(RESULTS_DIR, f"fold{fold_idx}_{head}_cm.png")
            cm = save_cm(labels, preds, f"Fold {fold_idx} - {head.title()} CM", cm_path, labels=(0,1), cmap=cmap)
            per_fold_images.append(cm_path)
            if head == "text":
                sum_cm_text += cm
            elif head == "audio":
                sum_cm_audio += cm
            else:
                sum_cm_fusion += cm

        # Per-language Text Head CMs
        imgs_lang, cm_lang_sum = per_language_cm_text(final_val, fold_idx, out_dir=RESULTS_DIR)
        per_fold_images.extend(imgs_lang)
        sum_cm_text_en += cm_lang_sum["english"]
        sum_cm_text_hi += cm_lang_sum["hindi"]

        # Text vs Audio agreement CM
        p_tvsa, cm_tvsa = text_vs_audio_cm(final_val, fold_idx, out_dir=RESULTS_DIR)
        per_fold_images.append(p_tvsa)
        sum_cm_text_vs_audio += cm_tvsa

        # ------ Aggregate per-fold metrics (global) ------
        t_acc, t_rec, t_f1, t_prec = compute_metrics(final_val["t_labels"], final_val["t_preds"])
        a_acc, a_rec, a_f1, a_prec = compute_metrics(final_val["a_labels"], final_val["a_preds"])
        f_acc, f_rec, f_f1, f_prec = compute_metrics(final_val["f_labels"], final_val["f_preds"])

        metrics_acc["text"].append(t_acc); metrics_rec["text"].append(t_rec); metrics_f1["text"].append(t_f1); metrics_prec["text"].append(t_prec)
        metrics_acc["audio"].append(a_acc); metrics_rec["audio"].append(a_rec); metrics_f1["audio"].append(a_f1); metrics_prec["audio"].append(a_prec)
        metrics_acc["fusion"].append(f_acc); metrics_rec["fusion"].append(f_rec); metrics_f1["fusion"].append(f_f1); metrics_prec["fusion"].append(f_prec)

        # ------ Fusion ROC AUC & PR AUC + curves ------
        try:
            roc_auc = roc_auc_score(final_val["f_labels"], final_val["f_probs"])
        except ValueError:
            roc_auc = float('nan')
        try:
            pr_auc  = average_precision_score(final_val["f_labels"], final_val["f_probs"])
        except ValueError:
            pr_auc = float('nan')

        fusion_roc_aucs.append(roc_auc)
        fusion_pr_aucs.append(pr_auc)

        # Plot ROC
        try:
            fpr, tpr, _ = roc_curve(final_val["f_labels"], final_val["f_probs"])
            plt.figure()
            plt.plot(fpr, tpr, label=f"Fold {fold_idx} (AUC={roc_auc:.3f})")
            plt.plot([0,1], [0,1], linestyle='--')
            plt.xlabel("False Positive Rate")
            plt.ylabel("True Positive Rate")
            plt.title(f"ROC Curve - Fold {fold_idx} (Fusion)")
            plt.legend(loc="lower right")
            roc_path = os.path.join(RESULTS_DIR, f"roc_fold{fold_idx}.png")
            plt.savefig(roc_path, bbox_inches="tight")
            plt.close()
            per_fold_images.append(roc_path)

            # interpolate TPR for mean curve
            tpr_interp = np.interp(mean_fpr, fpr, tpr)
            tpr_interp[0] = 0.0
            roc_tprs.append(tpr_interp)
        except Exception:
            pass

        # Plot PR
        try:
            precs, recalls, _ = precision_recall_curve(final_val["f_labels"], final_val["f_probs"])
            plt.figure()
            plt.plot(recalls, precs, label=f"Fold {fold_idx} (AP={pr_auc:.3f})")
            plt.xlabel("Recall")
            plt.ylabel("Precision")
            plt.title(f"Precision-Recall Curve - Fold {fold_idx} (Fusion)")
            plt.legend(loc="lower left")
            pr_path = os.path.join(RESULTS_DIR, f"pr_fold{fold_idx}.png")
            plt.savefig(pr_path, bbox_inches="tight")
            plt.close()
            per_fold_images.append(pr_path)

            # interpolate precision at common recall grid for mean PR curve
            prec_interp = np.interp(pr_recalls, recalls[::-1], precs[::-1])
            pr_precisions_interp.append(prec_interp)
        except Exception:
            pass

        # Save row for final report table
        per_fold_metric_table.append({
            "text_acc": t_acc, "text_rec": t_rec, "text_f1": t_f1, "text_prec": t_prec,
            "audio_acc": a_acc, "audio_rec": a_rec, "audio_f1": a_f1, "audio_prec": a_prec,
            "fusion_acc": f_acc, "fusion_rec": f_rec, "fusion_f1": f_f1, "fusion_prec": f_prec,
            "fusion_roc_auc": roc_auc, "fusion_pr_auc": pr_auc,
            "gate_mean": g_mean, "gate_var": g_var
        })

    # ===== Final Summary (Averaged) =====
    avg_text  = (np.mean(metrics_acc["text"]),  np.mean(metrics_rec["text"]),  np.mean(metrics_f1["text"]),  np.mean(metrics_prec["text"]))
    avg_audio = (np.mean(metrics_acc["audio"]), np.mean(metrics_rec["audio"]), np.mean(metrics_f1["audio"]), np.mean(metrics_prec["audio"]))
    avg_fuse  = (np.mean(metrics_acc["fusion"]),np.mean(metrics_rec["fusion"]),np.mean(metrics_f1["fusion"]),np.mean(metrics_prec["fusion"]))

    std_text  = (np.std(metrics_acc["text"]),  np.std(metrics_rec["text"]),  np.std(metrics_f1["text"]),  np.std(metrics_prec["text"]))
    std_audio = (np.std(metrics_acc["audio"]), np.std(metrics_rec["audio"]), np.std(metrics_f1["audio"]), np.std(metrics_prec["audio"]))
    std_fuse  = (np.std(metrics_acc["fusion"]),np.std(metrics_rec["fusion"]),np.std(metrics_f1["fusion"]),np.std(metrics_prec["fusion"]))

    # Plot averaged/overall confusion matrices as images
    final_images = []
    avg_tvsa_path = os.path.join(RESULTS_DIR, "avg_text_vs_audio_cm.png")
    averaged_cm_image(sum_cm_text_vs_audio, "Averaged Text vs Audio Agreement (5 folds)", avg_tvsa_path, labels=(0,1), cmap="Oranges")
    final_images.append(avg_tvsa_path)

    avg_en_path = os.path.join(RESULTS_DIR, "avg_text_english_cm.png")
    averaged_cm_image(sum_cm_text_en, "Averaged Text Head CM — English (5 folds)", avg_en_path, labels=(0,1), cmap="Greens")
    final_images.append(avg_en_path)

    avg_hi_path = os.path.join(RESULTS_DIR, "avg_text_hindi_cm.png")
    averaged_cm_image(sum_cm_text_hi, "Averaged Text Head CM — Hindi (5 folds)", avg_hi_path, labels=(0,1), cmap="Purples")
    final_images.append(avg_hi_path)

    # Overall (summed) CMs across folds for Text/Audio/Fusion
    overall_text_cm_path = os.path.join(RESULTS_DIR, "overall_text_cm.png")
    averaged_cm_image(sum_cm_text, "Overall Text CM (Summed Across Folds)", overall_text_cm_path, labels=(0,1), cmap="Greens")
    final_images.append(overall_text_cm_path)

    overall_audio_cm_path = os.path.join(RESULTS_DIR, "overall_audio_cm.png")
    averaged_cm_image(sum_cm_audio, "Overall Audio CM (Summed Across Folds)", overall_audio_cm_path, labels=(0,1), cmap="Purples")
    final_images.append(overall_audio_cm_path)

    overall_fusion_cm_path = os.path.join(RESULTS_DIR, "overall_fusion_cm.png")
    averaged_cm_image(sum_cm_fusion, "Overall Fusion CM (Summed Across Folds)", overall_fusion_cm_path, labels=(0,1), cmap="Blues")
    final_images.append(overall_fusion_cm_path)

    # Mean ROC & PR curves across folds (if available)
    if len(roc_tprs) > 0:
        mean_tpr = np.mean(roc_tprs, axis=0)
        mean_tpr[-1] = 1.0
        mean_auc = np.trapz(mean_tpr, mean_fpr)
        plt.figure()
        plt.plot(mean_fpr, mean_tpr, label=f"Mean ROC (AUC≈{mean_auc:.3f})")
        plt.plot([0,1], [0,1], linestyle='--')
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("Mean ROC Curve (Fusion)")
        plt.legend(loc="lower right")
        roc_mean_path = os.path.join(RESULTS_DIR, "roc_mean_across_folds.png")
        plt.savefig(roc_mean_path, bbox_inches="tight")
        plt.close()
        final_images.append(roc_mean_path)

    if len(pr_precisions_interp) > 0:
        mean_prec = np.mean(pr_precisions_interp, axis=0)
        mean_ap = np.trapz(mean_prec[::-1], pr_recalls[::-1])
        plt.figure()
        plt.plot(pr_recalls, mean_prec, label=f"Mean PR (AP≈{mean_ap:.3f})")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title("Mean Precision-Recall Curve (Fusion)")
        plt.legend(loc="lower left")
        pr_mean_path = os.path.join(RESULTS_DIR, "pr_mean_across_folds.png")
        plt.savefig(pr_mean_path, bbox_inches="tight")
        plt.close()
        final_images.append(pr_mean_path)

    # Render a small image with averaged global metrics table (with Prec)
    fig, ax = plt.subplots(figsize=(6.6, 3.2))
    ax.axis('off')
    collabels=("Head", "Accuracy (μ±σ)", "Recall (μ±σ)", "F1 (μ±σ)", "Precision (μ±σ)")
    table_data = [
        ["Text",  f"{avg_text[0]:.3f}±{std_text[0]:.3f}", f"{avg_text[1]:.3f}±{std_text[1]:.3f}", f"{avg_text[2]:.3f}±{std_text[2]:.3f}", f"{avg_text[3]:.3f}±{std_text[3]:.3f}"],
        ["Audio", f"{avg_audio[0]:.3f}±{std_audio[0]:.3f}", f"{avg_audio[1]:.3f}±{std_audio[1]:.3f}", f"{avg_audio[2]:.3f}±{std_audio[2]:.3f}", f"{avg_audio[3]:.3f}±{std_audio[3]:.3f}"],
        ["Fusion",f"{avg_fuse[0]:.3f}±{std_fuse[0]:.3f}", f"{avg_fuse[1]:.3f}±{std_fuse[1]:.3f}", f"{avg_fuse[2]:.3f}±{std_fuse[2]:.3f}", f"{avg_fuse[3]:.3f}±{std_fuse[3]:.3f}"],
    ]
    the_table = ax.table(cellText=table_data, colLabels=collabels, loc='center')
    the_table.scale(1, 1.6)
    metrics_img = os.path.join(RESULTS_DIR, "avg_global_metrics.png")
    plt.tight_layout()
    plt.savefig(metrics_img, bbox_inches="tight")
    plt.close()
    final_images.append(metrics_img)

    # (Optional) quick minimal PDF (per-epoch lines + images) to keep your original pipeline intact
    try:
        export_pdf(per_epoch_rows, per_fold_images, final_images, out_path=os.path.join(RESULTS_DIR, "summary_quick.pdf"))
    except Exception as e:
        print(f"export_pdf failed (continuing): {e}")

    # ===== Rich final PDF =====
    try:
        generate_final_report(
            out_file=os.path.join(RESULTS_DIR, "final_report.pdf"),
            n_folds=N_FOLDS,
            gate_means=np.array(gate_means),
            gate_vars=np.array(gate_vars),
            fusion_precisions=np.array(metrics_prec["fusion"]),
            fusion_roc_aucs=np.array(fusion_roc_aucs),
            fusion_pr_aucs=np.array(fusion_pr_aucs),
            per_fold_metric_table=per_fold_metric_table
        )
    except Exception as e:
        print(f"generate_final_report failed: {e}")


Loading pretrained encoders...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Epoch 13 | Val Acc=0.9100 | Val Rec=0.8400 | Val F1=0.9032 | Val Prec=0.9767


/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

Epoch 14 | Val Acc=0.9600 | Val Rec=0.9800 | Val F1=0.9608 | Val Prec=0.9423
⏹️ Early stopping triggered.
Loaded best checkpoint: results/checkpoints/fold5_best.pt


/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

PDF saved at results/summary_quick.pdf
✅ Final PDF report saved at results/final_report.pdf
